# Ph-negative MPN Classification — Multi-Seed Ensemble (Kaggle)

Monolithic notebook: **Baseline**, **CNN+PSO**, **CNN+GWO**, **Hybrid PSO-GWO** with:

- Multi-seed training (`SEED_LIST = [42, 123, 777, 1024, 2024]`)
- **Ensemble soft voting** on the held-out test set
- **Mean ± Std** reporting across seeds (robustness)
- **Grad-CAM** from the checkpoint with **lowest validation loss**

**Kaggle setup:** Add your dataset (folders `PV/`, `ET/`, `MF/`) as a notebook input.  
Enable **GPU** (Settings → Accelerator → GPU). Checkpoints save under `/kaggle/working/`.

## 1. Imports & Kaggle paths

In [ ]:
import json
import random
import ssl
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from tensorflow.keras.utils import img_to_array, load_img

ssl._create_default_https_context = ssl._create_unverified_context

# ------------------------------------------------------------------
# Kaggle-aware paths (falls back to local ./dataset when not on Kaggle)
# ------------------------------------------------------------------
KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

# >>> MANUAL OVERRIDE (only if auto-detect fails):
# Root must contain PV/, ET/, MF/ (not the ET/ folder itself).
# DATASET_DIR_OVERRIDE = Path("/kaggle/input/datasets/feyete0/dataset")
DATASET_DIR_OVERRIDE = None

REQUIRED_CLASSES = ("PV", "ET", "MF")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def _has_class_layout(root: Path) -> bool:
    """True if root contains PV/, ET/, MF/ (case-insensitive) with at least one image each."""
    if not root.is_dir():
        return False
    found = 0
    for cname in REQUIRED_CLASSES:
        # exact name first, then case-insensitive match among child dirs
        candidates = [root / cname]
        if not candidates[0].is_dir():
            candidates = [p for p in root.iterdir() if p.is_dir() and p.name.upper() == cname]
        ok = False
        for cdir in candidates:
            if not cdir.is_dir():
                continue
            if any(p.suffix.lower() in IMAGE_EXTS for p in cdir.iterdir() if p.is_file()):
                ok = True
                break
        if ok:
            found += 1
    return found == len(REQUIRED_CLASSES)


def _find_dataset_roots(search_roots, max_depth: int = 6) -> list[Path]:
    """Recursively collect directories that look like PV/ET/MF dataset roots."""
    hits: list[Path] = []

    def walk(base: Path, depth: int) -> None:
        if depth > max_depth or not base.is_dir():
            return
        if _has_class_layout(base):
            hits.append(base.resolve())
            return  # do not descend further once a valid root is found here
        try:
            children = sorted(base.iterdir())
        except OSError:
            return
        for child in children:
            if child.is_dir() and not child.name.startswith("."):
                walk(child, depth + 1)

    for sr in search_roots:
        if sr.is_dir():
            walk(sr.resolve(), 0)
    # prefer shallowest paths (closest to /kaggle/input/<dataset>/)
    return sorted(set(hits), key=lambda p: (len(p.parts), str(p)))


def _kaggle_datasets_api_roots() -> list[Path]:
    """
    Kaggle 'Datasets' tab layout (2024+), e.g.:
      /kaggle/input/datasets/feyete0/dataset/ET/et_....png
    Root for the notebook = .../dataset/  (parent of PV, ET, MF).
    """
    base = KAGGLE_INPUT / "datasets"
    if not base.is_dir():
        return []
    hits: list[Path] = []
    try:
        for user_dir in sorted(base.iterdir()):
            if not user_dir.is_dir():
                continue
            for ds_dir in sorted(user_dir.iterdir()):
                if not ds_dir.is_dir():
                    continue
                nested = ds_dir / "dataset"
                if nested.is_dir() and _has_class_layout(nested):
                    hits.append(nested.resolve())
                elif _has_class_layout(ds_dir):
                    hits.append(ds_dir.resolve())
    except OSError:
        pass
    return sorted(set(hits), key=lambda p: (len(p.parts), str(p)))


def _list_kaggle_input_hint() -> str:
    if not KAGGLE_INPUT.is_dir():
        return "( /kaggle/input not mounted — not running on Kaggle? )"
    lines = ["Contents of /kaggle/input:"]
    try:
        for item in sorted(KAGGLE_INPUT.iterdir()):
            if item.is_dir():
                subs = [x.name for x in sorted(item.iterdir())[:12]]
                lines.append(f"  {item.name}/  →  {subs}{' ...' if len(subs) >= 12 else ''}")
            else:
                lines.append(f"  {item.name}")
        api = _kaggle_datasets_api_roots()
        if api:
            lines.append("Detected dataset API roots (PV/ET/MF):")
            for p in api[:5]:
                lines.append(f"  → {p}")
    except OSError as e:
        lines.append(f"  (could not list: {e})")
    return "\n".join(lines)


def resolve_dataset_dir() -> Path:
    """Locate PV/ET/MF class folders under /kaggle/input, override, or local ./dataset."""
    if DATASET_DIR_OVERRIDE is not None:
        override = Path(DATASET_DIR_OVERRIDE)
        if _has_class_layout(override):
            return override.resolve()
        raise FileNotFoundError(
            f"DATASET_DIR_OVERRIDE set but invalid layout: {override}\n"
            f"Expected subfolders {REQUIRED_CLASSES} each containing images.\n"
            f"(Your image path should look like: {override}/ET/some_image.png)"
        )

    # Fast path: /kaggle/input/datasets/<user>/<name>/dataset/
    api_hits = _kaggle_datasets_api_roots()
    if api_hits:
        if len(api_hits) > 1:
            print("[info] Multiple Kaggle dataset roots; using:", api_hits[0])
        return api_hits[0]

    search_roots: list[Path] = []
    local = Path("dataset")
    if local.is_dir():
        search_roots.append(local)
    if KAGGLE_INPUT.is_dir():
        search_roots.append(KAGGLE_INPUT)
        if (KAGGLE_INPUT / "datasets").is_dir():
            search_roots.append(KAGGLE_INPUT / "datasets")
        search_roots.extend(p for p in KAGGLE_INPUT.iterdir() if p.is_dir())

    hits = _find_dataset_roots(search_roots)
    if hits:
        chosen = hits[0]
        if len(hits) > 1:
            print("[info] Multiple dataset roots found; using:", chosen)
            print("       Alternatives:", *hits[1:4], sep="\n         ")
        return chosen

    raise FileNotFoundError(
        "Could not find dataset with PV/ET/MF subfolders containing images.\n\n"
        "Fix on Kaggle:\n"
        "  1. Notebook → Add Input → select your dataset (must expose PV/, ET/, MF/).\n"
        "  2. Or set DATASET_DIR_OVERRIDE = Path('/kaggle/input/datasets/feyete0/dataset')\n"
        "     (parent of PV/, ET/, MF/ — not the ET/ folder itself).\n\n"
        + _list_kaggle_input_hint()
    )


DATASET_DIR = resolve_dataset_dir()
PROJECT_ROOT = WORKING_DIR
print(f"Dataset : {DATASET_DIR}")
print(f"Outputs : {PROJECT_ROOT}")
print(f"GPU     : {tf.config.list_physical_devices('GPU')}")

# cuFFT/cuDNN "already registered" lines are harmless TensorFlow/XLA noise on Kaggle — safe to ignore.

## 2. Configuration & multi-seed list

In [ ]:
# --- Experiment configuration ---
CLASS_NAMES = ("PV", "ET", "MF")
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = (224, 224)
BACKBONE = "ResNet50"

BASELINE_LR = 1e-3
BASELINE_DROPOUT = 0.5
BASELINE_BATCH = 32

TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20

META_POPULATION = 5
META_ITERATIONS = 5
META_FITNESS_EPOCHS = 4
FINAL_EPOCHS = 30

LR_MIN, LR_MAX = 1e-5, 1e-2
DROPOUT_MIN, DROPOUT_MAX = 0.2, 0.6
BATCH_CHOICES = (16, 32)

# Multi-seed list (replaces single global seed)
SEED_LIST = [42, 123, 777, 1024, 2024]

ARCHITECTURES = ("baseline", "pso", "gwo", "hybrid")
DISPLAY_NAMES = {
    "baseline": "Baseline ResNet50",
    "pso": "CNN + PSO",
    "gwo": "CNN + GWO",
    "hybrid": "Hybrid PSO-GWO",
}


def set_global_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def weights_path(arch: str, seed: int) -> Path:
    return PROJECT_ROOT / f"weights_{arch}_seed_{seed}.weights.h5"

## 3. Data pipeline (`prepare_datasets`)

In [ ]:
def _resolve_class_dir(root: Path, class_name: str) -> Path | None:
    exact = root / class_name
    if exact.is_dir():
        return exact
    for p in root.iterdir():
        if p.is_dir() and p.name.upper() == class_name.upper():
            return p
    return None


def _collect_image_paths(root: Path):
    paths, labels = [], []
    for idx, name in enumerate(CLASS_NAMES):
        class_dir = _resolve_class_dir(root, name)
        if class_dir is None:
            continue
        for p in sorted(class_dir.iterdir()):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                paths.append(str(p))
                labels.append(idx)
    if not paths:
        raise FileNotFoundError(f"No images under {root} with {CLASS_NAMES}")
    return paths, labels


def get_preprocess_fn():
    if BACKBONE == "ResNet50":
        return tf.keras.applications.resnet50.preprocess_input
    if BACKBONE == "DenseNet121":
        return tf.keras.applications.densenet.preprocess_input
    if BACKBONE == "MobileNetV2":
        return tf.keras.applications.mobilenet_v2.preprocess_input
    raise ValueError(BACKBONE)


def _make_augment_fn():
    return tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip("horizontal_and_vertical"),
            tf.keras.layers.RandomRotation(0.35),
            tf.keras.layers.RandomZoom((-0.35, 0.35), (-0.35, 0.35), fill_mode="reflect"),
            tf.keras.layers.RandomContrast(0.35),
            tf.keras.layers.RandomBrightness(0.35, value_range=(0.0, 255.0)),
        ],
        name="heavy_augmentation",
    )


def prepare_datasets(batch_size: int, seed: int):
    """Stratified 80/20 test split; 80/20 train/val within the train pool."""
    paths, labels = _collect_image_paths(DATASET_DIR)
    paths = np.array(paths)
    labels = np.array(labels, dtype=np.int32)

    tv_paths, test_paths, tv_labels, test_labels = train_test_split(
        paths, labels, test_size=TEST_SIZE, stratify=labels, random_state=seed
    )
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        tv_paths, tv_labels, test_size=VAL_SIZE_WITHIN_TRAIN,
        stratify=tv_labels, random_state=seed,
    )

    preprocess = get_preprocess_fn()
    augment = _make_augment_fn()

    def _decode_resize(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, IMG_SIZE)
        return tf.cast(img, tf.float32), label

    def train_map(path, label):
        img, label = _decode_resize(path, label)
        img = augment(img, training=True)
        return preprocess(img), label

    def eval_map(path, label):
        img, label = _decode_resize(path, label)
        return preprocess(img), label

    autotune = tf.data.AUTOTUNE
    train_ds = (
        tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
        .shuffle(len(train_paths), seed=seed, reshuffle_each_iteration=True)
        .map(train_map, num_parallel_calls=autotune)
        .batch(batch_size).prefetch(autotune)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
        .map(eval_map, num_parallel_calls=autotune)
        .batch(batch_size).prefetch(autotune)
    )
    test_ds = (
        tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
        .map(eval_map, num_parallel_calls=autotune)
        .batch(batch_size).prefetch(autotune)
    )
    meta = {
        "n_train": len(train_paths), "n_val": len(val_paths), "n_test": len(test_paths),
        "test_paths": test_paths, "test_labels": test_labels,
    }
    return train_ds, val_ds, test_ds, meta

## 4. Model builder (ResNet50 transfer learning)

In [ ]:
def build_model(learning_rate: float, dropout_rate: float) -> tf.keras.Model:
    tf.keras.backend.clear_session()
    kwargs = dict(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3),
                  pooling=None, name="backbone")
    if BACKBONE == "ResNet50":
        base = tf.keras.applications.ResNet50(**kwargs)
    elif BACKBONE == "DenseNet121":
        base = tf.keras.applications.DenseNet121(**kwargs)
    elif BACKBONE == "MobileNetV2":
        base = tf.keras.applications.MobileNetV2(**kwargs)
    else:
        raise ValueError(BACKBONE)
    base.trainable = False

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(128, activation="relu", name="fc_hidden")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
    model = tf.keras.Model(inputs, outputs, name=f"mpn_{BACKBONE.lower()}")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model

## 5. Meta-heuristics (PSO, GWO, Hybrid) — seed-aware

In [ ]:
def _decode_particle(vec):
    v = np.clip(vec.astype(np.float64), 0.0, 1.0)
    lr = LR_MIN * (LR_MAX / LR_MIN) ** v[0]
    dr = DROPOUT_MIN + v[1] * (DROPOUT_MAX - DROPOUT_MIN)
    bs = BATCH_CHOICES[0] if v[2] < 0.5 else BATCH_CHOICES[1]
    return float(lr), float(dr), int(bs)


def _train_short(lr, dropout, batch_size, epochs, split_seed, verbose=0):
    train_ds, val_ds, _, _ = prepare_datasets(batch_size=batch_size, seed=split_seed)
    model = build_model(lr, dropout)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=verbose)
    val_loss = float(hist.history["val_loss"][-1])
    del model; tf.keras.backend.clear_session()
    return val_loss


def optimize_pso(split_seed, meta_seed, verbose=0):
    rng = np.random.default_rng(meta_seed)
    pop, iters, fit_ep = META_POPULATION, META_ITERATIONS, META_FITNESS_EPOCHS
    dim, lo, hi = 3, np.zeros(3), np.ones(3)
    w, c1, c2, v_max = 0.72, 1.49, 1.49, 0.2
    X = rng.uniform(lo, hi, (pop, dim))
    V = rng.uniform(-v_max, v_max, (pop, dim))
    pbest, pbest_f = X.copy(), np.full(pop, np.inf)
    trace = []
    for gen in range(iters):
        gbest = pbest[int(np.argmin(pbest_f))].copy()
        for i in range(pop):
            if gen > 0:
                r1, r2 = rng.random(dim), rng.random(dim)
                V[i] = np.clip(w * V[i] + c1 * r1 * (pbest[i] - X[i]) + c2 * r2 * (gbest - X[i]), -v_max, v_max)
                X[i] = np.clip(X[i] + V[i], lo, hi)
            lr, dr, bs = _decode_particle(X[i])
            f = _train_short(lr, dr, bs, fit_ep, split_seed, verbose)
            if f < pbest_f[i]:
                pbest_f[i], pbest[i] = f, X[i].copy()
        trace.append(float(np.min(pbest_f)))
    best_vec = pbest[int(np.argmin(pbest_f))]
    lr, dr, bs = _decode_particle(best_vec)
    return {"learning_rate": lr, "dropout": dr, "batch_size": bs}, trace


def optimize_gwo(split_seed, meta_seed, verbose=0):
    rng = np.random.default_rng(meta_seed)
    pop, iters, fit_ep = META_POPULATION, META_ITERATIONS, META_FITNESS_EPOCHS
    dim, lo, hi = 3, np.zeros(3), np.ones(3)
    X = rng.uniform(lo, hi, (pop, dim))

    def eval_pack(positions):
        return np.array([_train_short(*_decode_particle(p), fit_ep, split_seed, verbose) for p in positions])

    fitness = eval_pack(X)
    order = np.argsort(fitness)
    alpha, beta, delta = X[order[0]].copy(), X[order[1]].copy(), X[order[2]].copy()
    trace = [float(fitness[order[0]])]
    for t in range(max(iters - 1, 0)):
        a = 2.0 * (1.0 - (t + 1) / float(max(iters, 1)))
        for i in range(pop):
            r1, r2 = rng.random(dim), rng.random(dim)
            A1, C1 = 2 * a * r1 - a, 2 * r2
            X1 = alpha - A1 * np.abs(C1 * alpha - X[i])
            r1, r2 = rng.random(dim), rng.random(dim)
            A2, C2 = 2 * a * r1 - a, 2 * r2
            X2 = beta - A2 * np.abs(C2 * beta - X[i])
            r1, r2 = rng.random(dim), rng.random(dim)
            A3, C3 = 2 * a * r1 - a, 2 * r2
            X3 = delta - A3 * np.abs(C3 * delta - X[i])
            X[i] = np.clip((X1 + X2 + X3) / 3.0, lo, hi)
        fitness = eval_pack(X)
        order = np.argsort(fitness)
        alpha, beta, delta = X[order[0]].copy(), X[order[1]].copy(), X[order[2]].copy()
        trace.append(float(fitness[order[0]]))
    lr, dr, bs = _decode_particle(alpha)
    return {"learning_rate": lr, "dropout": dr, "batch_size": bs}, trace


def optimize_hybrid_pso_gwo(split_seed, meta_seed, verbose=0):
    rng = np.random.default_rng(meta_seed)
    pop, iters, fit_ep = META_POPULATION, META_ITERATIONS, META_FITNESS_EPOCHS
    dim, lo, hi = 3, np.zeros(3), np.ones(3)
    w, c1, c2, c3, v_max = 0.72, 1.35, 1.35, 0.85, 0.25
    X = rng.uniform(lo, hi, (pop, dim))
    V = rng.uniform(-v_max, v_max, (pop, dim))
    pbest, pbest_f = X.copy(), np.full(pop, np.inf)
    trace = []
    for gen in range(iters):
        fitness = np.empty(pop)
        for i in range(pop):
            lr, dr, bs = _decode_particle(X[i])
            fitness[i] = _train_short(lr, dr, bs, fit_ep, split_seed, verbose)
            if fitness[i] < pbest_f[i]:
                pbest_f[i], pbest[i] = fitness[i], X[i].copy()
        trace.append(float(np.min(pbest_f)))
        if gen == iters - 1:
            break
        order = np.argsort(fitness)
        alpha, beta, delta = X[order[0]].copy(), X[order[1]].copy(), X[order[2]].copy()
        a = 2.0 * (1.0 - gen / float(max(iters - 1, 1))) if iters > 1 else 2.0
        for i in range(pop):
            r1, r2 = rng.random(dim), rng.random(dim)
            X1 = alpha - (2 * a * r1 - a) * np.abs(2 * rng.random(dim) * alpha - X[i])
            r1, r2 = rng.random(dim), rng.random(dim)
            X2 = beta - (2 * a * r1 - a) * np.abs(2 * rng.random(dim) * beta - X[i])
            r1, r2 = rng.random(dim), rng.random(dim)
            X3 = delta - (2 * a * r1 - a) * np.abs(2 * rng.random(dim) * delta - X[i])
            X_gwo = (X1 + X2 + X3) / 3.0
            rp1, rp2, rp3 = rng.random(dim), rng.random(dim), rng.random(dim)
            V[i] = np.clip(w * V[i] + c1 * rp1 * (pbest[i] - X[i]) + c2 * rp2 * (alpha - X[i]) + c3 * rp3 * (X_gwo - X[i]), -v_max, v_max)
            X[i] = np.clip(X[i] + V[i], lo, hi)
    best_vec = pbest[int(np.argmin(pbest_f))]
    lr, dr, bs = _decode_particle(best_vec)
    return {"learning_rate": lr, "dropout": dr, "batch_size": bs}, trace

## 6. Training, metrics & ensemble helpers

In [ ]:
def history_to_dict(hist):
    h = hist.history
    acc = h.get("accuracy", h.get("sparse_categorical_accuracy"))
    val_acc = h.get("val_accuracy", h.get("sparse_categorical_accuracy"))
    return {
        "loss": list(map(float, h["loss"])),
        "val_loss": list(map(float, h["val_loss"])),
        "accuracy": list(map(float, acc)),
        "val_accuracy": list(map(float, val_acc)),
    }


def train_and_save(arch, hp, split_seed, verbose=1):
    """Full training; returns history dict, min val_loss, test probabilities."""
    train_ds, val_ds, test_ds, meta = prepare_datasets(hp["batch_size"], split_seed)
    model = build_model(hp["learning_rate"], hp["dropout"])
    hist = model.fit(train_ds, validation_data=val_ds, epochs=FINAL_EPOCHS, verbose=verbose)
    hdict = history_to_dict(hist)
    min_val_loss = float(np.min(hdict["val_loss"]))
    proba = model.predict(test_ds, verbose=0)
    wpath = weights_path(arch, split_seed)
    model.save_weights(str(wpath))
    del model; tf.keras.backend.clear_session()
    return hdict, min_val_loss, proba, meta, wpath


def macro_specificity(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    specs = []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return float(np.mean(specs))


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "sensitivity": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "specificity": macro_specificity(y_true, y_pred),
        "f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def ensemble_soft_vote(proba_list):
    """Mean probability across seeds → argmax classification."""
    mean_proba = np.mean(np.stack(proba_list, axis=0), axis=0)
    return mean_proba, np.argmax(mean_proba, axis=1)


def fmt_mean_std(values):
    arr = np.asarray(values, dtype=np.float64)
    return f"{arr.mean():.4f} ± {arr.std(ddof=1):.4f}"

## 7. Grad-CAM (Keras 3-safe, lowest-val-loss checkpoint)

In [ ]:
def _inner_backbone(model):
    try:
        bb = model.get_layer("backbone")
        if isinstance(bb, tf.keras.Model):
            return bb
    except ValueError:
        pass
    for lay in model.layers:
        if isinstance(lay, tf.keras.Model):
            return lay
    raise ValueError("No backbone found")


def _spatial_tensor(model):
    gap = model.get_layer("gap")
    inp = gap.input[0] if isinstance(gap.input, (list, tuple)) else gap.input
    if BACKBONE != "ResNet50":
        return inp
    bb = _inner_backbone(model)
    if not any(l.name == "conv5_block3_out" for l in bb.layers):
        return inp
    conv_out = bb.get_layer("conv5_block3_out").output
    try:
        tf.keras.Model(model.input, [conv_out, model.output])
        return conv_out
    except ValueError:
        return inp


def compute_gradcam(model, img_batch, pred_index=None):
    spatial = _spatial_tensor(model)
    grad_model = tf.keras.Model(model.input, [spatial, model.output], name="gradcam")
    batch = tf.cast(img_batch[:1], tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(batch, training=False)
        tape.watch(conv_out)
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]).numpy())
        score = preds[:, pred_index]
    grads = tape.gradient(score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    heat = tf.nn.relu(tf.reduce_sum(pooled * conv_out, axis=-1))
    heat = tf.squeeze(heat, 0)
    heat = heat / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()


def overlay_gradcam(display_01, heatmap, alpha=0.45):
    h, w = display_01.shape[:2]
    heat = tf.image.resize(np.expand_dims(heatmap, -1), (h, w)).numpy().squeeze()
    jet = plt.get_cmap("jet")(np.clip(heat, 0, 1))[..., :3].astype(np.float32)
    return np.clip(jet * alpha + display_01 * (1 - alpha), 0, 1)


def save_gradcam_for_path(image_path, model, out_path, true_class=None):
    img = load_img(image_path, target_size=IMG_SIZE)
    arr = img_to_array(img).astype(np.float32)
    disp = arr / 255.0
    batch = np.expand_dims(get_preprocess_fn()(arr.copy()), axis=0)
    pred_idx = int(np.argmax(model.predict(batch, verbose=0)[0]))
    heat = compute_gradcam(model, batch, pred_index=pred_idx)
    over = overlay_gradcam(disp, heat)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(disp); ax[0].set_title("Input"); ax[0].axis("off")
    ax[1].imshow(over)
    title = f"Grad-CAM — pred: {CLASS_NAMES[pred_idx]}"
    if true_class:
        title += f" | true: {true_class}"
    ax[1].set_title(title); ax[1].axis("off")
    plt.tight_layout(); plt.savefig(out_path, dpi=200, bbox_inches="tight"); plt.close()


def pick_one_test_image_per_class(meta, pick_seed):
    paths, labels = np.asarray(meta["test_paths"]), np.asarray(meta["test_labels"])
    rng = np.random.default_rng(pick_seed)
    out = {}
    for ci, cname in enumerate(CLASS_NAMES):
        idxs = np.where(labels == ci)[0]
        out[cname] = str(paths[int(rng.choice(idxs))])
    return out

## 8. Multi-seed training loop

For each seed: stratified split → PSO/GWO/Hybrid search → final training of all four architectures.  
Checkpoints: `weights_{arch}_seed_{seed}.weights.h5` in `/kaggle/working/`.

In [ ]:
# Storage across seeds
seed_results = {}          # seed -> {arch: {hp, history, min_val_loss, proba, wpath}}
per_seed_metrics = {a: [] for a in ARCHITECTURES}  # single-model test metrics per seed
ensemble_probas = {a: [] for a in ARCHITECTURES}
y_true_ref = None
meta_ref = None

for seed in SEED_LIST:
    print("\n" + "=" * 72)
    print(f"SEED {seed}")
    print("=" * 72)
    set_global_seeds(seed)

    # --- Meta-heuristic search (meta RNG offset per algorithm) ---
    print("[PSO] search...")
    pso_hp, pso_trace = optimize_pso(split_seed=seed, meta_seed=seed, verbose=0)
    print(f"  best: {pso_hp}")

    print("[GWO] search...")
    gwo_hp, gwo_trace = optimize_gwo(split_seed=seed, meta_seed=seed + 7, verbose=0)
    print(f"  best: {gwo_hp}")

    print("[Hybrid] search...")
    hyb_hp, hyb_trace = optimize_hybrid_pso_gwo(split_seed=seed, meta_seed=seed + 101, verbose=0)
    print(f"  best: {hyb_hp}")

    tf.keras.backend.clear_session()

    hp_map = {
        "baseline": {"learning_rate": BASELINE_LR, "dropout": BASELINE_DROPOUT, "batch_size": BASELINE_BATCH},
        "pso": pso_hp,
        "gwo": gwo_hp,
        "hybrid": hyb_hp,
    }

    seed_results[seed] = {}
    for arch in ARCHITECTURES:
        print(f"\n--- Final training: {DISPLAY_NAMES[arch]} (seed={seed}) ---")
        hdict, min_vl, proba, meta, wpath = train_and_save(arch, hp_map[arch], seed, verbose=1)
        if y_true_ref is None:
            y_true_ref = np.asarray(meta["test_labels"], dtype=np.int32)
            meta_ref = meta
        y_pred = np.argmax(proba, axis=1)
        m = compute_metrics(y_true_ref, y_pred)
        per_seed_metrics[arch].append(m)
        ensemble_probas[arch].append(proba)
        seed_results[seed][arch] = {
            "hp": hp_map[arch], "history": hdict, "min_val_loss": min_vl,
            "proba": proba, "wpath": str(wpath), "metrics": m,
        }
        print(f"  min_val_loss={min_vl:.4f}  test_acc={m['accuracy']:.4f}  saved→ {wpath.name}")

# Persist run manifest for reproducibility
manifest = {
    "seed_list": SEED_LIST,
    "architectures": list(ARCHITECTURES),
    "seed_results": {
        str(s): {a: {"hp": seed_results[s][a]["hp"], "min_val_loss": seed_results[s][a]["min_val_loss"],
                     "metrics": seed_results[s][a]["metrics"], "wpath": seed_results[s][a]["wpath"]}
                 for a in ARCHITECTURES}
        for s in SEED_LIST
    },
}
manifest_path = PROJECT_ROOT / "ensemble_run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"\nManifest saved: {manifest_path}")

## 9. Ensemble soft voting & statistical reporting (Mean ± Std)

In [ ]:
# --- Per-seed single-model robustness (Mean ± Std) ---
robust_rows = []
for arch in ARCHITECTURES:
    accs = [m["accuracy"] for m in per_seed_metrics[arch]]
    f1s = [m["f1"] for m in per_seed_metrics[arch]]
    sens = [m["sensitivity"] for m in per_seed_metrics[arch]]
    robust_rows.append({
        "Model": DISPLAY_NAMES[arch],
        "Accuracy (Mean ± Std)": fmt_mean_std(accs),
        "F1-Score (Mean ± Std)": fmt_mean_std(f1s),
        "Sensitivity (Mean ± Std)": fmt_mean_std(sens),
        "Accuracy (per seed)": [round(x, 4) for x in accs],
        "F1 (per seed)": [round(x, 4) for x in f1s],
    })

df_robust = pd.DataFrame(robust_rows)
print("\n=== Single-model robustness across seeds (test set) ===\n")
print(df_robust[["Model", "Accuracy (Mean ± Std)", "F1-Score (Mean ± Std)", "Sensitivity (Mean ± Std)"]].to_string(index=False))
df_robust.to_csv(PROJECT_ROOT / "ensemble_robustness_mean_std.csv", index=False)

# --- Ensemble soft voting (primary evaluation) ---
ensemble_rows = []
ensemble_proba_dict = {}
for arch in ARCHITECTURES:
    mean_proba, y_pred_ens = ensemble_soft_vote(ensemble_probas[arch])
    m = compute_metrics(y_true_ref, y_pred_ens)
    ensemble_proba_dict[DISPLAY_NAMES[arch]] = mean_proba
    ensemble_rows.append({
        "Model": DISPLAY_NAMES[arch] + " (Ensemble)",
        "Accuracy": m["accuracy"],
        "Sensitivity (Recall)": m["sensitivity"],
        "Specificity": m["specificity"],
        "F1-Score": m["f1"],
    })
    cm = confusion_matrix(y_true_ref, y_pred_ens, labels=list(range(NUM_CLASSES)))
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.title(f"Ensemble CM — {DISPLAY_NAMES[arch]}")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / f"ensemble_cm_{arch}.png", dpi=200); plt.close()

df_ensemble = pd.DataFrame(ensemble_rows)
print("\n=== Ensemble soft-voting (test set) ===\n")
print(df_ensemble.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
df_ensemble.to_csv(PROJECT_ROOT / "ensemble_soft_vote_metrics.csv", index=False)

# Combined ROC (ensemble probabilities)
y_bin = label_binarize(y_true_ref, classes=list(range(NUM_CLASSES)))
plt.figure(figsize=(9, 7))
for name, proba in ensemble_proba_dict.items():
    fpr, tpr, _ = roc_curve(y_bin.ravel(), proba.ravel())
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Ensemble ROC — micro-averaged")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(PROJECT_ROOT / "ensemble_combined_ROC.png", dpi=200); plt.close()
print("\nSaved: ensemble_robustness_mean_std.csv, ensemble_soft_vote_metrics.csv, ensemble_cm_*.png")

## 10. Grad-CAM — checkpoint with **lowest validation loss**

Selects the `(architecture, seed)` pair with minimum `min_val_loss` among Hybrid runs (proposed model).  
Change `GRADCAM_ARCH` to `"baseline"` etc. if needed.

In [ ]:
GRADCAM_ARCH = "hybrid"  # proposed architecture for XAI

best_seed = min(SEED_LIST, key=lambda s: seed_results[s][GRADCAM_ARCH]["min_val_loss"])
best_vl = seed_results[best_seed][GRADCAM_ARCH]["min_val_loss"]
best_hp = seed_results[best_seed][GRADCAM_ARCH]["hp"]
best_wpath = weights_path(GRADCAM_ARCH, best_seed)

print(f"Grad-CAM architecture : {DISPLAY_NAMES[GRADCAM_ARCH]}")
print(f"Best seed (min val_loss={best_vl:.4f}) : {best_seed}")
print(f"Loading weights       : {best_wpath}")

set_global_seeds(best_seed)
xai_model = build_model(best_hp["learning_rate"], best_hp["dropout"])
xai_model.load_weights(str(best_wpath))

pick_seed = best_seed + 777
per_class = pick_one_test_image_per_class(meta_ref, pick_seed)
for cname, img_path in per_class.items():
    out = PROJECT_ROOT / f"gradcam_{GRADCAM_ARCH}_seed{best_seed}_{cname}.png"
    save_gradcam_for_path(img_path, xai_model, out, true_class=cname)
    print(f"Saved Grad-CAM → {out.name}")

del xai_model; tf.keras.backend.clear_session()
print("\nDone. Download outputs from /kaggle/working/ (Output tab).")